### *** ML-Mini Project ***

>> *~HARSHITHA. B. T PES1UG24AM118*


## Experiment 03: NGA-West2 Dataset Inspection

### Objective
Inspect the NGA-West2 flatfile structure, identify metadata fields relevant to the earthquake directionality pipeline, and examine missing-value conventions before joining records to waveform data.

### Observations
The inspected `d005` flatfile contained 21,540 rows and 274 columns. The inspection covered earthquake magnitude, Joyner–Boore distance, Vs30, PGA, PGV, and horizontal-record references. The data-quality checks also identified the use of `-999` as a missing-value sentinel in multiple fields.

### Relevance to the pipeline
This inspection helps determine which metadata fields are available for record matching and subsequent feature construction. The horizontal-record references and component azimuth fields are particularly relevant to identifying usable waveform pairs.

### Limitations
This notebook inspects the selected flatfile; it does not establish that every row is usable for modelling. Missing-value handling, record matching, and physical validity must be checked in the subsequent pipeline. The inspected files are a subset of NGA-West2, not the complete database.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/Earthquake_ML_MiniProject"
)

print("Folder exists:", ROOT.exists())

if ROOT.exists():
    files = [p for p in ROOT.rglob("*") if p.is_file()]
    print("Total files found:", len(files))

    for p in files[:50]:
        print(p.relative_to(ROOT))
else:
    print("Check the folder name and Drive location.")

Folder exists: True
Total files found: 940
Updated_NGA_West2_flatfiles_part1/Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx
Updated_NGA_West2_flatfiles_part1/Updated_NGA_West2_Flatfile_RotD50_d010_public_version.xlsx
Updated_NGA_West2_flatfiles_part1/Updated_NGA_West2_Flatfile_RotD50_d020_public_version.xlsx
Updated_NGA_West2_flatfiles_part1/Updated_NGA_West2_Flatfile_RotD50_d030_public_version.xlsx
Updated_NGA_West2_flatfiles_part1/Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx
Updated_NGA_West2_flatfiles_part1/Updated_NGA_West2_Flatfile_RotD50_d070_public_version.xlsx
Updated_NGA_West2_flatfiles_part2/Updated_NGA_West2_Flatfile_RotD50_d100_public_version.xlsx
Updated_NGA_West2_flatfiles_part2/Updated_NGA_West2_Flatfile_RotD50_d150_public_version.xlsx
Updated_NGA_West2_flatfiles_part2/Updated_NGA_West2_Flatfile_RotD50_d200_public_version.xlsx
Updated_NGA_West2_flatfiles_part2/Updated_NGA_West2_Flatfile_RotD50_d250_public_version.xlsx
Updated_NGA_West2_flatfiles

In [4]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/Earthquake_ML_MiniProject"
)

for folder in [
    "Updated_NGA_West2_flatfiles_part1",
    "Updated_NGA_West2_flatfiles_part2"
]:
    folder_path = ROOT / folder

    print("\n" + "=" * 90)
    print("FOLDER:", folder)
    print("Exists:", folder_path.exists())

    if folder_path.exists():
        excel_paths = sorted(folder_path.glob("*.xlsx"))
        print("Excel workbooks:", len(excel_paths))

        for path in excel_paths:
            print("\n" + "-" * 70)
            print("FILE:", path.name)

            try:
                xls = pd.ExcelFile(path)
                print("Sheets:", xls.sheet_names)

                for sheet in xls.sheet_names[:2]:
                    df = pd.read_excel(
                        path,
                        sheet_name=sheet,
                        nrows=3
                    )

                    print(f"\nSheet: {sheet}")
                    print("Columns:", len(df.columns))
                    print(df.columns.tolist())
                    ## display(df.head(3))
                    # The selected d005 flatfile is inspected in detail below.

            except Exception as e:
                print("ERROR:", repr(e))
    else:
        print("Folder not found at this location.")


FOLDER: Updated_NGA_West2_flatfiles_part1
Exists: True
Excel workbooks: 6

----------------------------------------------------------------------
FILE: Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx
Sheets: ['ROTD50_d005']

Sheet: ROTD50_d005
Columns: 274
['Record Sequence Number', 'EQID', 'Earthquake Name', 'YEAR', 'MODY', 'HRMN', 'Station Name', 'Station Sequence Number', 'Station ID  No.', 'Earthquake Magnitude', 'Magnitude Type', 'Magnitude Uncertainty: Kagan Model', 'Magnitude Uncertainty: Statistical', 'Magnitude Sample Size', 'Magnitude Uncertainty: Study Class', 'Mo (dyne.cm)', 'Strike (deg)', 'Dip (deg)', 'Rake Angle (deg)', 'Mechanism Based on Rake Angle', 'P-plunge (deg)', 'P-trend (deg)', 'T-plunge (deg)', 'T-trend (deg)', 'Hypocenter Latitude (deg)', 'Hypocenter Longitude (deg)', 'Hypocenter Depth (km)', 'Coseismic Surface Rupture: 1=Yes; 0=No;    -999=Unknown', 'Coseismic Surface Rupture (Including Inferred)', 'Basis for Inference of Surface Rupture', 'Finite

In [5]:
from pathlib import Path
import pandas as pd
import warnings
import time

warnings.filterwarnings(
    "ignore",
    message="Unknown extension is not supported and will be removed",
    category=UserWarning
)

ROOT = Path(
    "/content/drive/MyDrive/Earthquake_ML_MiniProject"
)

folders = [
    ROOT / "Updated_NGA_West2_flatfiles_part1",
    ROOT / "Updated_NGA_West2_flatfiles_part2"
]

excel_paths = sorted(
    p for folder in folders
    for p in folder.glob("*.xlsx")
)

print("Excel files found:", len(excel_paths))
for p in excel_paths:
    print(p.name, "|", round(p.stat().st_size / 1e6, 2), "MB")

print("\nInspecting d005 only...")
d005_path = next(
    p for p in excel_paths if "_d005_" in p.name
)

start = time.time()

df_d005 = pd.read_excel(
    d005_path,
    sheet_name=0,
    engine="openpyxl"
)

print("Loaded successfully!")
print("Shape:", df_d005.shape)
print("Time taken:", round(time.time() - start, 2), "seconds")
print("\nFirst 5 rows:")
display(df_d005.head())

print("\nKey columns available:")
for col in [
    "EQID",
    "Earthquake Magnitude",
    "Station Name",
    "Joyner-Boore Dist. (km)",
    "PGA (g)",
    "Idirectivity",
    "Tp",
    "Ry 2"
]:
    print(f"{col}: {col in df_d005.columns}")

Excel files found: 11
Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx | 51.2 MB
Updated_NGA_West2_Flatfile_RotD50_d010_public_version.xlsx | 51.07 MB
Updated_NGA_West2_Flatfile_RotD50_d020_public_version.xlsx | 50.98 MB
Updated_NGA_West2_Flatfile_RotD50_d030_public_version.xlsx | 50.91 MB
Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx | 50.23 MB
Updated_NGA_West2_Flatfile_RotD50_d070_public_version.xlsx | 50.83 MB
Updated_NGA_West2_Flatfile_RotD50_d100_public_version.xlsx | 50.72 MB
Updated_NGA_West2_Flatfile_RotD50_d150_public_version.xlsx | 50.63 MB
Updated_NGA_West2_Flatfile_RotD50_d200_public_version.xlsx | 50.55 MB
Updated_NGA_West2_Flatfile_RotD50_d250_public_version.xlsx | 50.48 MB
Updated_NGA_West2_Flatfile_RotD50_d300_public_version.xlsx | 50.39 MB

Inspecting d005 only...
Loaded successfully!
Shape: (21540, 274)
Time taken: 46.55 seconds

First 5 rows:


,Record Sequence Number,EQID,Earthquake Name,YEAR,MODY,HRMN,Station Name,Station Sequence Number,Station ID No.,Earthquake Magnitude,...,Instrument Nat. Freq.,Instrument Damping,Instrument Type,Quality_Flag,Spectra Quality Flag,Late S-trigger,Late P-trigger,Idirectivity,Tp,Ry 2
0,1,1,"Helena, Montana-01",1935,1031,1838,Carroll College,197,2022,6.0,...,-999.0,-999.0,-999,-999,0,-999,-999,0,-999.0,8.046386
1,2,2,"Helena, Montana-02",1935,1031,1918,Helena Fed Bldg,198,2229,6.0,...,-999.0,-999.0,-999,-999,0,-999,-999,0,-999.0,0.155745
2,3,3,Humbolt Bay,1937,207,442,Ferndale City Hall,133,1023,5.8,...,-999.0,-999.0,-999,-999,0,-999,-999,0,-999.0,-38.820722
3,4,4,Imperial Valley-01,1938,606,242,El Centro Array #9,75,117,5.0,...,-999.0,-999.0,-999,-999,0,-999,-999,0,-999.0,23.958844
4,5,5,Northwest Calif-01,1938,912,610,Ferndale City Hall,133,1023,5.5,...,-999.0,-999.0,-999,-999,0,-999,-999,0,-999.0,51.112544



Key columns available:
EQID: True
Earthquake Magnitude: True
Station Name: True
Joyner-Boore Dist. (km): True
PGA (g): True
Idirectivity: True
Tp: True
Ry 2: True


In [6]:
d005_path = next(
    p for p in excel_paths if "_d005_" in p.name
)

df_d005 = pd.read_excel(d005_path, sheet_name=0)

important_columns = [
    "Record Sequence Number",
    "EQID",
    "Earthquake Name",
    "Earthquake Magnitude",
    "Station Name",
    "Station ID  No.",
    "Joyner-Boore Dist. (km)",
    "Vs30 (m/s) selected for analysis",
    "PGA (g)",
    "PGV (cm/sec)",
    "Idirectivity",
    "Tp",
    "Ry 2"
]

available_columns = [
    col for col in important_columns
    if col in df_d005.columns
]

print("d005 shape:", df_d005.shape)
print("\nSelected column summary:")
display(df_d005[available_columns].describe(include="all").T)

print("\nMissing-value counts:")
display(
    df_d005[available_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

print("\nRows containing the -999 sentinel:")
for col in available_columns:
    if pd.api.types.is_numeric_dtype(df_d005[col]):
        print(
            f"{col}:",
            int((df_d005[col] == -999).sum())
        )

d005 shape: (21540, 274)

Selected column summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Record Sequence Number,21540.0,NaN,NaN,NaN,10770.5,6218.206735,1.0,5385.75,10770.5,16155.25,21540.0
EQID,21540.0,NaN,NaN,NaN,733.178087,442.747704,-999.0,180.0,1018.0,1096.0,1266.0
Earthquake Name,21540,608,Chuetsu-oki,616,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Earthquake Magnitude,21540.0,NaN,NaN,NaN,4.39775,23.724029,-999.0,3.9,4.5,6.2,7.9
Station Name,21540,4161,Palomar,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Station ID No.,21540.0,3039.0,-999.0,2901.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Joyner-Boore Dist. (km),21540.0,NaN,NaN,NaN,112.949361,115.341657,-999.0,44.54,89.135,158.875,1532.66
Vs30 (m/s) selected for analysis,21540.0,NaN,NaN,NaN,466.500653,203.720757,-999.0,337.9425,416.52,572.53,2100.0
PGA (g),21540.0,NaN,NaN,NaN,-9.43288,96.764056,-999.0,0.000336,0.002617,0.019121,1.7683
PGV (cm/sec),21540.0,NaN,NaN,NaN,-6.747611,97.375799,-999.0,0.014131,0.118695,1.769275,256.62



Missing-value counts:


,missing_count
Record Sequence Number,0
EQID,0
Earthquake Name,0
Earthquake Magnitude,0
Station Name,0
Station ID No.,0
Joyner-Boore Dist. (km),0
Vs30 (m/s) selected for analysis,0
PGA (g),0
PGV (cm/sec),0



Rows containing the -999 sentinel:
Record Sequence Number: 0
EQID: 11
Earthquake Magnitude: 12
Joyner-Boore Dist. (km): 46
Vs30 (m/s) selected for analysis: 25
PGA (g): 204
PGV (cm/sec): 204
Idirectivity: 12935
Tp: 21391
Ry 2: 349


In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Earthquake_ML_MiniProject/"
    "experiment_03_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

columns_to_check = [
    "EQID",
    "Earthquake Magnitude",
    "Joyner-Boore Dist. (km)",
    "Vs30 (m/s) selected for analysis",
    "PGA (g)",
    "PGV (cm/sec)",
    "Idirectivity",
    "Tp",
    "Ry 2"
]

quality_rows = []

for col in columns_to_check:
    values = df_d005[col]
    numeric = pd.to_numeric(values, errors="coerce")

    is_sentinel = numeric.eq(-999)
    valid = numeric[~is_sentinel & numeric.notna()]

    quality_rows.append({
        "column": col,
        "total_rows": len(values),
        "actual_NaN": int(values.isna().sum()),
        "sentinel_minus999": int(is_sentinel.sum()),
        "valid_after_sentinel_removal": int(valid.notna().sum()),
        "valid_after_sentinel_removal": int(valid.count()),
        "valid_min": valid.min() if len(valid) else np.nan,
        "valid_median": valid.median() if len(valid) else np.nan,
        "valid_max": valid.max() if len(valid) else np.nan
    })

quality_df = pd.DataFrame(quality_rows)

display(quality_df)

quality_df.to_csv(
    OUTPUT_DIR / "d005_data_quality_report.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "d005_data_quality_report.csv")

,column,total_rows,actual_NaN,sentinel_minus999,valid_after_sentinel_removal,valid_min,valid_median,valid_max
0,EQID,21540,0,11,21529,1.000000e+00,1018.000000,1266.000000
1,Earthquake Magnitude,21540,0,12,21528,2.990000e+00,4.500000,7.900000
2,Joyner-Boore Dist. (km),21540,0,46,21494,0.000000e+00,89.365000,1532.660000
3,Vs30 (m/s) selected for analysis,21540,0,25,21515,8.932000e+01,416.520000,2100.000000
4,PGA (g),21540,0,204,21336,3.653200e-07,0.002729,1.768300
5,PGV (cm/sec),21540,0,204,21336,7.387900e-06,0.124770,256.620000
6,Idirectivity,21540,0,12935,8605,0.000000e+00,0.000000,1.000000
7,Tp,21540,0,21391,149,3.220000e-01,4.417000,13.118000
8,Ry 2,21540,0,349,21191,-1.461400e+03,5.816007,814.239107


Saved: /content/drive/MyDrive/Earthquake_ML_MiniProject/experiment_03_outputs/d005_data_quality_report.csv


In [8]:
for col in ["Idirectivity", "Tp", "Ry 2"]:
    values = pd.to_numeric(df_d005[col], errors="coerce")
    valid = values[values.notna() & values.ne(-999)]

    print("\n" + "=" * 55)
    print("COLUMN:", col)
    print("Valid values:", len(valid), "of", len(values))
    print("Unique valid values:", valid.nunique())
    print("Smallest values:")
    print(valid.nsmallest(10).to_list())
    print("Largest values:")
    print(valid.nlargest(10).to_list())


COLUMN: Idirectivity
Valid values: 8605 of 21540
Unique valid values: 2
Smallest values:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Largest values:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

COLUMN: Tp
Valid values: 149 of 21540
Unique valid values: 135
Smallest values:
[0.322, 0.518, 0.574, 0.588, 0.7, 0.728, 0.7896, 0.805, 0.84, 0.854]
Largest values:
[13.118, 12.621, 12.313, 12.285, 12.019, 11.956, 10.633, 10.395, 10.381, 10.318]

COLUMN: Ry 2
Valid values: 21191 of 21540
Unique valid values: 21110
Smallest values:
[-1461.4001, -1427.8651, -1416.0718, -1414.1053, -1410.0401, -1383.4001, -1375.3097, -1341.643, -1329.0422, -1315.9808]
Largest values:
[814.239107134949, 676.622807801945, 637.17087, 558.82005, 536.28112, 535.88303, 521.9858, 520.928560275249, 518.27398, 517.82037]


In [9]:
# Experiment 3 — inspect columns and data types
# Reuses df_d005 already loaded in memory.

print("Dataset shape:", df_d005.shape)

print("\nAll columns:")
for i, col in enumerate(df_d005.columns):
    print(f"{i:03d}: {col} | dtype={df_d005[col].dtype}")

print("\nPotential directionality / rotation-related columns:")
keywords = [
    "rot", "direct", "azimuth", "angle", "orientation",
    "strike", "rupture", "fault", "component", "period",
    "direction"
]

for col in df_d005.columns:
    if any(word in col.lower() for word in keywords):
        print(col)

Dataset shape: (21540, 274)

All columns:
000: Record Sequence Number | dtype=int64
001: EQID | dtype=int64
002: Earthquake Name | dtype=object
003: YEAR | dtype=int64
004: MODY | dtype=int64
005: HRMN | dtype=int64
006: Station Name | dtype=object
007: Station Sequence Number | dtype=int64
008: Station ID  No. | dtype=object
009: Earthquake Magnitude | dtype=float64
010: Magnitude Type | dtype=object
011: Magnitude Uncertainty: Kagan Model | dtype=float64
012: Magnitude Uncertainty: Statistical | dtype=float64
013: Magnitude Sample Size | dtype=int64
014: Magnitude Uncertainty: Study Class | dtype=float64
015: Mo (dyne.cm) | dtype=float64
016: Strike (deg) | dtype=float64
017: Dip (deg) | dtype=float64
018: Rake Angle (deg) | dtype=int64
019: Mechanism Based on Rake Angle | dtype=int64
020: P-plunge (deg) | dtype=float64
021: P-trend (deg) | dtype=float64
022: T-plunge (deg) | dtype=float64
023: T-trend (deg) | dtype=float64
024: Hypocenter Latitude (deg) | dtype=float64
025: Hypocent

In [10]:
# Identify spectral-period columns and check their numerical coverage.

spectral_cols = [
    col for col in df_d005.columns
    if col.startswith("T") and col.endswith("S")
]

print("Number of spectral columns:", len(spectral_cols))
print("First 15:", spectral_cols[:15])
print("Last 15:", spectral_cols[-15:])

if spectral_cols:
    print("\nSample spectral values:")
    print(df_d005[spectral_cols[:5]].head())

    print("\nSentinel counts in the first five spectral columns:")
    print((df_d005[spectral_cols[:5]] == -999).sum())

Number of spectral columns: 111
First 15: ['T0.010S', 'T0.020S', 'T0.022S', 'T0.025S', 'T0.029S', 'T0.030S', 'T0.032S', 'T0.035S', 'T0.036S', 'T0.040S', 'T0.042S', 'T0.044S', 'T0.045S', 'T0.046S', 'T0.048S']
Last 15: ['T6.000S', 'T6.500S', 'T7.000S', 'T7.500S', 'T8.000S', 'T8.500S', 'T9.000S', 'T9.500S', 'T10.000S', 'T11.000S', 'T12.000S', 'T13.000S', 'T14.000S', 'T15.000S', 'T20.000S']

Sample spectral values:
    T0.010S   T0.020S   T0.022S   T0.025S   T0.029S
0  0.157476  0.164311  0.165726  0.162331  0.170577
1  0.046460  0.051758  0.048996  0.051178  0.058713
2  0.040995  0.041169  0.041347  0.041342  0.041617
3  0.018448  0.019238  0.018758  0.018822  0.019758
4  0.122504  0.125593  0.122460  0.124287  0.125755

Sentinel counts in the first five spectral columns:
T0.010S    204
T0.020S    204
T0.022S    204
T0.025S    204
T0.029S    204
dtype: int64


In [11]:
# Experiment 3B — inspect horizontal-record references

cols = [
    "Record Sequence Number",
    "EQID",
    "Earthquake Name",
    "Earthquake Magnitude",
    "Station Name",
    "Station ID  No.",
    "Joyner-Boore Dist. (km)",
    "File Name (Horizontal 1)",
    "File Name (Horizontal 2)",
    "H1 azimth (degrees)",
    "H2 azimith (degrees)",
]

# Use only columns that exist in this workbook.
cols = [c for c in cols if c in df_d005.columns]

print("Available columns:", cols)
display(df_d005[cols].head(10).to_string(index=False))

print("\nHorizontal filename coverage:")
for col in ["File Name (Horizontal 1)", "File Name (Horizontal 2)"]:
    if col in df_d005.columns:
        values = df_d005[col].astype(str).str.strip()
        valid = ~values.isin(["", "-999", "nan", "None"])
        print(f"{col}: {valid.sum()} / {len(df_d005)} populated")
        print("Examples:", values[valid].head(5).tolist())

Available columns: ['Record Sequence Number', 'EQID', 'Earthquake Name', 'Earthquake Magnitude', 'Station Name', 'Station ID  No.', 'Joyner-Boore Dist. (km)', 'File Name (Horizontal 1)', 'File Name (Horizontal 2)', 'H1 azimth (degrees)', 'H2 azimith (degrees)']


' Record Sequence Number  EQID    Earthquake Name  Earthquake Magnitude       Station Name Station ID  No.  Joyner-Boore Dist. (km) File Name (Horizontal 1) File Name (Horizontal 2)  H1 azimth (degrees)  H2 azimith (degrees)\n                      1     1 Helena, Montana-01                  6.00    Carroll College            2022                     2.07    HELENA.A\\A-HMC180.AT2    HELENA.A\\A-HMC270.AT2                  180                   270\n                      2     2 Helena, Montana-02                  6.00    Helena Fed Bldg            2229                     2.09    HELENA.B\\B-FEB000.AT2    HELENA.B\\B-FEB090.AT2                    0                    90\n                      3     3        Humbolt Bay                  5.80 Ferndale City Hall            1023                    71.28       HUMBOLT\\FRN225.AT2       HUMBOLT\\FRN315.AT2                  225                   315\n                      4     4 Imperial Valley-01                  5.00 El Centro Array #9    


Horizontal filename coverage:
File Name (Horizontal 1): 21512 / 21540 populated
Examples: ['HELENA.A\\A-HMC180.AT2', 'HELENA.B\\B-FEB000.AT2', 'HUMBOLT\\FRN225.AT2', 'IMPVALL.BG\\B-ELC000.AT2', 'NWCALIF.AB\\A-FRN045.AT2']
File Name (Horizontal 2): 21512 / 21540 populated
Examples: ['HELENA.A\\A-HMC270.AT2', 'HELENA.B\\B-FEB090.AT2', 'HUMBOLT\\FRN315.AT2', 'IMPVALL.BG\\B-ELC090.AT2', 'NWCALIF.AB\\A-FRN135.AT2']
